# Topic 4: Pipeline Monitoring & Observability

> **Phase 7 → Week 14 → Topic 4**

---

## Why Monitoring Matters

A pipeline that runs but produces silently wrong data is worse than one that fails loudly. Production monitoring has three layers:

```
Infrastructure:  Is the cluster running? Are nodes healthy? (CloudWatch, YARN UI)
Job-level:       Did the job succeed? How long did it take? (Airflow, Spark History)
Data-level:      Is the output correct? Row counts, null rates, freshness (custom checks)
```

---

## Interview Cheat Sheet

**Q: What metrics would you monitor for a production Spark ETL pipeline?**
> Job-level: run duration, success/failure rate, SLA compliance. Spark-level: shuffle read/write bytes (spill indicator), GC time (> 10% of task time = memory pressure), executor lost events (node failures), task retry count (skew/OOM indicator). Data-level: output row count vs expected range, null rate on key columns, data freshness (max event_time vs wall clock), referential integrity. Alert on: job failure, duration > 2× baseline, row count deviation > 20%, null rate > threshold.

**Q: What is the Spark History Server and what can you find there?**
> The Spark History Server provides a UI for completed Spark applications using event logs stored on HDFS/S3. You can see: job/stage/task breakdown, execution timeline, executor utilization, shuffle bytes, spill to disk, GC time, skew (tasks with widely varying durations in the same stage). It's the primary tool for diagnosing slow jobs post-mortem. Enable with `spark.eventLog.enabled=true` and `spark.eventLog.dir=s3://bucket/spark-logs/`.

**Q: What is data freshness and how do you monitor it?**
> Data freshness measures the lag between when events occur and when they appear in the data warehouse/lake. Monitor by: comparing `MAX(event_timestamp)` in the output table to `NOW()`. Alert if lag exceeds SLA (e.g., Gold table must reflect events within 2 hours). Freshness degradation is often the first signal of upstream ingestion issues before jobs start failing.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import os, shutil, json, time

spark = SparkSession.builder \
    .appName("Week14 - Pipeline Monitoring") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
BASE = "/tmp/monitoring_demo"
if os.path.exists(BASE): shutil.rmtree(BASE)
os.makedirs(BASE)
print("Pipeline Monitoring — ready")

## Part 1: Job-Level Metrics (Airflow + CloudWatch)

In [ ]:
print("""
Job-Level Monitoring Patterns:
══════════════════════════════════════════════════════════════

1. Airflow metrics (built-in):
   Airflow emits StatsD metrics → Prometheus → Grafana

   Key metrics:
   airflow.dag.daily_etl.duration           → job wall time
   airflow.task.silver_transform.success    → success count
   airflow.task.silver_transform.failed     → failure count
   airflow.scheduler.tasks.pending          → queue backlog
   airflow.executor.open_slots              → available worker slots

   Enable StatsD:
   [metrics]
   statsd_on = True
   statsd_host = prometheus-statsd-exporter
   statsd_port = 8125

2. Custom CloudWatch metrics from Airflow task:
   import boto3
   from datetime import datetime

   def publish_metrics(**context):
       cw = boto3.client('cloudwatch', region_name='us-east-1')
       duration = (datetime.now() - context['task_instance'].start_date).seconds
       cw.put_metric_data(
           Namespace='DataPipeline/ETL',
           MetricData=[
               {'MetricName': 'JobDuration',
                'Value': duration,
                'Unit': 'Seconds',
                'Dimensions': [
                    {'Name': 'DagId',  'Value': context['dag'].dag_id},
                    {'Name': 'TaskId', 'Value': context['task_instance'].task_id}
                ]},
               {'MetricName': 'OutputRowCount',
                'Value': context['ti'].xcom_pull(key='row_count'),
                'Unit': 'Count',
                'Dimensions': [{'Name': 'Layer', 'Value': 'Silver'}]}
           ]
       )
       print(f"Published metrics: duration={duration}s")

3. CloudWatch Alarms:
   # Alert if job duration > 2 hours
   cw.put_metric_alarm(
       AlarmName='ETL-Duration-High',
       MetricName='JobDuration',
       Namespace='DataPipeline/ETL',
       Period=86400,              # 1 day
       EvaluationPeriods=1,
       Threshold=7200,            # 2 hours
       ComparisonOperator='GreaterThanThreshold',
       AlarmActions=['arn:aws:sns:us-east-1:123:PagerDuty'],
   )
══════════════════════════════════════════════════════════════
""")

## Part 2: Data Quality Monitoring

In [ ]:
# Build a simple data quality metrics framework
from pyspark.sql import DataFrame
from typing import Dict, List
import json

def compute_dq_metrics(df: DataFrame, table_name: str, key_columns: List[str]) -> Dict:
    """Compute data quality metrics for a DataFrame."""
    total_rows = df.count()
    metrics = {
        "table": table_name,
        "timestamp": datetime.now().isoformat(),
        "total_rows": total_rows,
        "columns": {}
    }

    for col in key_columns:
        null_count  = df.filter(F.col(col).isNull()).count()
        null_rate   = null_count / total_rows if total_rows > 0 else 0
        distinct    = df.select(col).distinct().count()
        metrics["columns"][col] = {
            "null_count":  null_count,
            "null_rate":   round(null_rate, 4),
            "distinct":    distinct,
        }
        # Add numeric stats for numeric columns
        if dict(df.dtypes)[col] in ("double", "int", "long", "float"):
            stats = df.select(
                F.min(col).alias("min"),
                F.max(col).alias("max"),
                F.avg(col).alias("mean")
            ).collect()[0]
            metrics["columns"][col]["min"]  = stats["min"]
            metrics["columns"][col]["max"]  = stats["max"]
            metrics["columns"][col]["mean"] = round(stats["mean"] or 0, 2)

    return metrics


def check_dq_thresholds(metrics: Dict, thresholds: Dict) -> List[str]:
    """Return list of violations."""
    violations = []
    row_count = metrics["total_rows"]

    if "min_rows" in thresholds and row_count < thresholds["min_rows"]:
        violations.append(f"Row count {row_count} < min {thresholds['min_rows']}")
    if "max_rows" in thresholds and row_count > thresholds["max_rows"]:
        violations.append(f"Row count {row_count} > max {thresholds['max_rows']}")

    for col, col_thresholds in thresholds.get("columns", {}).items():
        col_metrics = metrics["columns"].get(col, {})
        if "max_null_rate" in col_thresholds:
            null_rate = col_metrics.get("null_rate", 0)
            if null_rate > col_thresholds["max_null_rate"]:
                violations.append(f"{col}: null_rate={null_rate:.2%} > max {col_thresholds['max_null_rate']:.2%}")
        if "min_value" in col_thresholds:
            if (col_metrics.get("min") or 0) < col_thresholds["min_value"]:
                violations.append(f"{col}: min={col_metrics.get('min')} < threshold {col_thresholds['min_value']}")

    return violations


# --- Demo ---
orders = spark.createDataFrame([
    ("O001", "C001", 250.0, "shipped"),
    ("O002", None,  -10.0, "pending"),   # bad: null customer, negative amount
    ("O003", "C002", 180.0, "shipped"),
    ("O004", "C001", 320.0, None),       # bad: null status
], ["order_id", "customer_id", "amount", "status"])

metrics = compute_dq_metrics(
    orders,
    table_name="silver.orders",
    key_columns=["order_id", "customer_id", "amount", "status"]
)
print("DQ Metrics:")
print(json.dumps(metrics, indent=2))

thresholds = {
    "min_rows": 1,
    "columns": {
        "customer_id": {"max_null_rate": 0.01},   # max 1% nulls
        "amount":      {"max_null_rate": 0.0, "min_value": 0.0},
        "status":      {"max_null_rate": 0.05},
    }
}

violations = check_dq_thresholds(metrics, thresholds)
if violations:
    print(f"\nDQ VIOLATIONS ({len(violations)}):")
    for v in violations:
        print(f"  ❌ {v}")
else:
    print("\n✅ All DQ checks passed")

## Part 3: Data Freshness Monitoring

In [ ]:
# Data freshness check pattern
from datetime import datetime, timezone

# Simulate a Silver table with event timestamps
now_utc = datetime.now(timezone.utc)

silver_orders = spark.createDataFrame([
    ("O001", (now_utc - timedelta(hours=1)).strftime("%Y-%m-%d %H:%M:%S")),
    ("O002", (now_utc - timedelta(hours=2)).strftime("%Y-%m-%d %H:%M:%S")),
    ("O003", (now_utc - timedelta(hours=1, minutes=30)).strftime("%Y-%m-%d %H:%M:%S")),
], ["order_id", "event_time"]) \
 .withColumn("event_time", F.to_timestamp("event_time"))

def check_freshness(df: DataFrame, timestamp_col: str, max_lag_hours: float, table_name: str):
    max_event_time = df.agg(F.max(timestamp_col)).collect()[0][0]
    if max_event_time is None:
        print(f"⚠️  {table_name}: No data found (table may be empty)")
        return

    lag_hours = (datetime.now() - max_event_time).total_seconds() / 3600

    status = "✅" if lag_hours <= max_lag_hours else "❌"
    print(f"{status} {table_name}: max event={max_event_time}, lag={lag_hours:.1f}h (SLA: {max_lag_hours}h)")

    return lag_hours

check_freshness(silver_orders, "event_time", max_lag_hours=2.0, table_name="silver.orders")
check_freshness(silver_orders, "event_time", max_lag_hours=0.5, table_name="gold.revenue")  # fails

print("""
Freshness monitoring in production:
  1. Add to the END of every Gold job as a post-check task in Airflow
  2. Publish to CloudWatch: MetricName=DataFreshness, Value=lag_hours
  3. Alarm if lag_hours > SLA threshold
  4. Different SLAs per layer:
     Bronze:  event time ≤ 15 min ago (near real-time ingest)
     Silver:  event time ≤ 1 hour ago
     Gold:    updated ≤ 2 hours after data window closes
""")

## Part 4: Spark History Server & Key Metrics

In [ ]:
print("""
Spark Metrics to Monitor in Production:
══════════════════════════════════════════════════════════════

Enable event logging (required for History Server):
  spark.eventLog.enabled=true
  spark.eventLog.dir=s3://my-bucket/spark-event-logs/
  spark.history.fs.logDirectory=s3://my-bucket/spark-event-logs/

Start History Server:
  $SPARK_HOME/sbin/start-history-server.sh
  → http://localhost:18080

Key metrics to look for:

1. Shuffle read/write bytes:
   High shuffle → consider repartitioning strategy or broadcast joins
   Shuffle spill to disk → executor memory too small

2. GC Time:
   > 10% of task time → memory pressure, too many objects
   Fix: increase executor.memory, use Kryo serialization,
        avoid collect() on large data

3. Task duration distribution in a stage:
   All tasks take ~same time → healthy
   1-2 tasks take 10× longer → data skew!
   Fix: salting, AQE skew join, repartition before groupBy

4. Executor lost events:
   In event log or Spark UI → node failure or OOM kill
   Fix: reduce executor memory, increase memoryOverhead, check OOM killer

5. Spill metrics:
   memoryBytesSpilled / diskBytesSpilled → shuffle spill
   Fix: increase shuffle.partitions, reduce partition size,
        increase spark.memory.fraction

EMR CloudWatch metrics (automatic):
  YARNMemoryAvailablePercentage  → memory headroom
  HDFSUtilization               → disk usage
  MRActiveNodes                 → healthy node count
  S3BytesRead / S3BytesWritten  → data movement to/from S3
  ContainerAllocated            → executor containers running
  ContainerPending              → containers waiting (queue backlog)

Prometheus + Grafana for Spark:
  Enable Prometheus sink:
  spark.conf.set("spark.ui.prometheus.enabled", "true")
  → /metrics/prometheus at the Spark driver UI

  Key Grafana panels:
  - Jobs/stages per minute
  - Executor active tasks vs total
  - JVM heap used vs max
  - Shuffle bytes/sec
══════════════════════════════════════════════════════════════
""")

## Part 5: Alerting & On-Call Patterns

In [ ]:
print("""
Alerting Runbook — Production Pipeline:
══════════════════════════════════════════════════════════════

Alert tiers:
  P1 (wake someone up):
    - Gold table not updated within SLA (business impact now)
    - Pipeline failure 3 runs in a row
    - Data loss detected (row count drops > 50% from baseline)

  P2 (Slack alert, fix within 4 hours):
    - Single pipeline failure (likely transient, retrying)
    - DQ violation: null rate > threshold
    - Job duration > 2× baseline (degrading)

  P3 (Slack notification, fix next business day):
    - Job duration 1.5× baseline
    - SLA miss (job finished late but data is correct)
    - Schema change detected

Runbook for common failures:

  OOM (OutOfMemoryError):
    1. Check Spark UI → executor lost events
    2. Check driver logs: "java.lang.OutOfMemoryError"
    3. If shuffle spill: increase shuffle.partitions
    4. If executor OOM: increase executor.memory or reduce executor.cores
    5. If driver OOM: avoid collect(), use show()/write() instead

  Slow job (> 2× baseline):
    1. Check Spark UI → Stages → look for straggler tasks
    2. If skew: check data distribution, apply salting or AQE
    3. If GC > 10%: reduce partition size or increase memory
    4. If shuffle spill: more partitions, less data per partition
    5. Check S3: many small input files? Run compaction first

  Missing data (row count < expected):
    1. Check source: did upstream deliver fewer records?
    2. Check for accidental filter/dedup that dropped records
    3. Check for schema change that caused parse failures
    4. Check dead-letter queue for rejected records
    5. Check job bookmark / watermark (incremental job missed a range?)

  Duplicate data (row count > expected):
    1. Was the job re-run without clearing checkpoint?
    2. Did partitionOverwriteMode not fire correctly?
    3. Was MERGE INTO condition correct (joining on unique key)?
    4. Did source system send duplicates?

Key principle: EVERY alert must have a runbook link.
               If there's no runbook, the alert will be ignored.
══════════════════════════════════════════════════════════════
""")

## Exercises

1. Write a `compute_dq_metrics()` function that also detects duplicate rows (by primary key), referential integrity violations (customer_id must exist in customers table), and value distribution shifts (compare current avg to a stored baseline avg).
2. Build an Airflow task that runs after the Gold build job, checks freshness (max event_time < 2 hours ago), and fails the DAG if freshness is violated. How would you make the threshold configurable per environment?
3. Design a Grafana dashboard for a Spark ETL pipeline. List the 8 most important panels and their data sources (CloudWatch, Airflow StatsD, Spark Prometheus, custom DQ metrics).
4. Write a CloudWatch alarm that triggers a PagerDuty page when the Silver table's row count for the current day is more than 30% below the 7-day moving average.
5. Your Spark job's P99 task duration in Stage 3 is 45 minutes while median is 2 minutes. This is a textbook skew case. Walk through exactly what you'd check in the Spark UI to confirm skew, how you'd identify the skewed key, and three approaches to fix it.